# 💧 Hydropower Energy Generation Forecast
**Goal:** Predict `hydropower_output_mw` using environmental and operational parameters.

**Features used:**
- `rainfall_mm` — rainfall in millimeters
- `temperature_c` — temperature in Celsius
- `humidity_percent` — relative humidity
- `river_flow_m3s` — river flow (cubic meters/sec)
- `reservoir_level_percent` — reservoir fill level
- `upstream_inflow_m3s` — upstream water inflow
- `sediment_load` — sediment in the water
- `turbine_efficiency` — turbine efficiency (0–1)

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

print('✅ All libraries imported successfully!')

## 2. Load & Explore the Dataset

In [ ]:
df = pd.read_csv('hydropower_dataset.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
df.describe().round(2)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of target variable
plt.figure(figsize=(8, 4))
sns.histplot(df['hydropower_output_mw'], bins=40, kde=True, color='steelblue')
plt.title('Distribution of Hydropower Output (MW)')
plt.xlabel('hydropower_output_mw')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 7))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

print('\nTop features correlated with hydropower_output_mw:')
print(corr['hydropower_output_mw'].sort_values(ascending=False))

In [ ]:
# Scatter plots of top features vs target
top_features = ['river_flow_m3s', 'upstream_inflow_m3s', 'reservoir_level_percent', 'turbine_efficiency']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feat in zip(axes.flatten(), top_features):
    ax.scatter(df[feat], df['hydropower_output_mw'], alpha=0.4, color='teal', s=15)
    ax.set_xlabel(feat)
    ax.set_ylabel('hydropower_output_mw')
    ax.set_title(f'{feat} vs Output')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Define features and target
FEATURES = [
    'rainfall_mm', 'temperature_c', 'humidity_percent',
    'river_flow_m3s', 'reservoir_level_percent',
    'upstream_inflow_m3s', 'sediment_load', 'turbine_efficiency'
]
TARGET = 'hydropower_output_mw'

X = df[FEATURES]
y = df[TARGET]

# Train / Test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')

## 5. Train Multiple Models & Compare

In [ ]:
models = {
    'Linear Regression' : Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    'Random Forest'     : Pipeline([('scaler', StandardScaler()), ('model', RandomForestRegressor(n_estimators=100, random_state=42))]),
    'Gradient Boosting' : Pipeline([('scaler', StandardScaler()), ('model', GradientBoostingRegressor(n_estimators=100, random_state=42))]),
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    results.append({'Model': name, 'MAE': round(mae,3), 'RMSE': round(rmse,3), 'R²': round(r2,4)})
    trained_models[name] = model
    print(f'{name:25s}  MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.4f}')

results_df = pd.DataFrame(results)
print('\n')
print(results_df.sort_values('R²', ascending=False).to_string(index=False))

## 6. Best Model — Detailed Evaluation

In [ ]:
# Pick best model by R²
best_name = results_df.sort_values('R²', ascending=False).iloc[0]['Model']
best_model = trained_models[best_name]
print(f'🏆 Best Model: {best_name}')

y_pred_best = best_model.predict(X_test)

# Actual vs Predicted plot
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_best, alpha=0.5, color='royalblue', s=20)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect fit')
plt.xlabel('Actual Output (MW)')
plt.ylabel('Predicted Output (MW)')
plt.title(f'Actual vs Predicted — {best_name}')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residual plot
residuals = y_test.values - y_pred_best
plt.figure(figsize=(7, 4))
plt.scatter(y_pred_best, residuals, alpha=0.5, color='coral', s=15)
plt.axhline(0, color='black', lw=1.5, linestyle='--')
plt.xlabel('Predicted Output (MW)')
plt.ylabel('Residual')
plt.title('Residual Plot')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importances (if tree-based)
if hasattr(best_model.named_steps['model'], 'feature_importances_'):
    importances = pd.Series(best_model.named_steps['model'].feature_importances_, index=FEATURES).sort_values(ascending=True)
    plt.figure(figsize=(8, 5))
    importances.plot(kind='barh', color='mediumseagreen')
    plt.title(f'Feature Importances — {best_name}')
    plt.xlabel('Importance Score')
    plt.tight_layout()
    plt.show()

## 7. Forecast on New Data (Example)

In [ ]:
# Simulate a 10-step forecast with slightly varying inputs
np.random.seed(0)
forecast_df = pd.DataFrame({
    'rainfall_mm'              : np.random.uniform(80, 200, 10),
    'temperature_c'            : np.random.uniform(15, 40, 10),
    'humidity_percent'         : np.random.uniform(50, 95, 10),
    'river_flow_m3s'           : np.random.uniform(100, 600, 10),
    'reservoir_level_percent'  : np.random.uniform(30, 90, 10),
    'upstream_inflow_m3s'      : np.random.uniform(100, 650, 10),
    'sediment_load'            : np.random.uniform(15, 70, 10),
    'turbine_efficiency'       : np.random.uniform(0.70, 0.84, 10),
})

forecast_scaled = forecast_df[FEATURES]
forecast_df['predicted_mw'] = best_model.predict(forecast_scaled).round(2)

plt.figure(figsize=(9, 4))
plt.plot(range(1, 11), forecast_df['predicted_mw'], marker='o', color='steelblue', linewidth=2)
plt.fill_between(range(1, 11), forecast_df['predicted_mw'] * 0.9,
                 forecast_df['predicted_mw'] * 1.1, alpha=0.2, color='steelblue', label='±10% band')
plt.xticks(range(1, 11))
plt.xlabel('Future Time Step')
plt.ylabel('Predicted Output (MW)')
plt.title('Hydropower Output Forecast (10 Steps)')
plt.legend()
plt.tight_layout()
plt.show()

print(forecast_df[['predicted_mw']].to_string())

## 8. Save Model & Scaler (.sav)

In [ ]:
# Save the full pipeline (scaler + model) as a single .sav file
joblib.dump(best_model, 'hydropower_model.sav')

print('✅ hydropower_model.sav — saved (pipeline: scaler + model)')
print(f'   Model : {type(best_model.named_steps["model"]).__name__}')

In [ ]:
# Quick sanity check — reload and predict one row
loaded_model = joblib.load('hydropower_model.sav')

sample = X_test.iloc[[0]]
print('Sample input:\n', sample.to_string())
print(f'\nPredicted  : {loaded_model.predict(sample)[0]:.3f} MW')
print(f'Actual     : {y_test.iloc[0]:.3f} MW')

---
### Summary
| File | Purpose |
|---|---|
| `hydropower_model.sav` | Full pipeline: scaler + model in one file |

Run `streamlit run app.py` to launch the forecast web app.